In [1]:
import logging
import sys

sys.path.insert(0, "Singer")
sys.path.insert(0, "research")
sys.path

['research',
 'Singer',
 '/Users/rebelraider/.pyenv/versions/3.9.20/lib/python39.zip',
 '/Users/rebelraider/.pyenv/versions/3.9.20/lib/python3.9',
 '/Users/rebelraider/.pyenv/versions/3.9.20/lib/python3.9/lib-dynload',
 '',
 '/Users/rebelraider/Library/Caches/pypoetry/virtualenvs/xlabs-hack-2024-GK-D5AqO-py3.9/lib/python3.9/site-packages']

In [2]:
from huggingface_hub import snapshot_download 
snapshot_download(repo_id="Cyanbox/Prompt-Singer")

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

'/Users/rebelraider/.cache/huggingface/hub/models--Cyanbox--Prompt-Singer/snapshots/4a9ad215081865e168df26ea4a54cc78e87378a6'

In [3]:
!curl 2ip.ru

77.238.141.243


In [2]:
from Singer.fairseq.checkpoint_utils import load_model_ensemble_and_task_from_hf_hub

models, cfg, task = load_model_ensemble_and_task_from_hf_hub(
    "Cyanbox/Prompt-Singer"
)

print(models)

/Users/rebelraider/Library/Caches/pypoetry/virtualenvs/xlabs-hack-2024-GK-D5AqO-py3.9/lib/python3.9/site-packages/fairscale/experimental/nn/offload.py:19: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  return torch.cuda.amp.custom_fwd(orig_func)  # type: ignore
/Users/rebelraider/Library/Caches/pypoetry/virtualenvs/xlabs-hack-2024-GK-D5AqO-py3.9/lib/python3.9/site-packages/fairscale/experimental/nn/offload.py:30: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  return torch.cuda.amp.custom_bwd(orig_func)  # type: ignore


Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

/Users/rebelraider/.cache/fairseq/models--Cyanbox--Prompt-Singer/snapshots/4a9ad215081865e168df26ea4a54cc78e87378a6/prompt-singer-flant5-large-finetuned/
['/Users/rebelraider/.cache/fairseq/models--Cyanbox--Prompt-Singer/snapshots/4a9ad215081865e168df26ea4a54cc78e87378a6/prompt-singer-flant5-large-finetuned/checkpoint_last.pt']
/Users/rebelraider/.cache/fairseq/models--Cyanbox--Prompt-Singer/snapshots/4a9ad215081865e168df26ea4a54cc78e87378a6
[]


In [7]:
from research.PromptSinger.dataset.tokenizer.soundstream.AudioTokenizer import AudioTokenizer
import torch

# Load the audio tokenizer with a checkpoint path
def load_audio_tokenizer(ckpt_path='/Users/rebelraider/.cache/huggingface/hub/models--Cyanbox--Prompt-Singer/snapshots/4a9ad215081865e168df26ea4a54cc78e87378a6/codec/ckpt_01135000.pth'):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    audio_tokenizer = AudioTokenizer(ckpt_path=ckpt_path, device=device)
    return audio_tokenizer

audio_tokenizer = load_audio_tokenizer()

/Users/rebelraider/Library/Caches/pypoetry/virtualenvs/xlabs-hack-2024-GK-D5AqO-py3.9/lib/python3.9/site-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/Users/rebelraider/Documents/Python projects/Hackatons/XLabs-Hack-2024/Singer/research/PromptSinger/dataset/tokenizer/soundstream/AudioTokenizer.py:47: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed t

In [1]:
import sys

sys.path.insert(0, "Singer")
sys.path.insert(0, "research")

import torch
import Singer
import research
import fairseq
from fairseq.checkpoint_utils import load_model_ensemble_and_task
from fairseq.models.text_to_speech.hub_interface import TTSHubInterface

def generate_voice(text, model_path, data_cfg_path):
    """
    Generates speech from input text using a pre-trained TTS model.

    Args:
        text (str): The input text to be converted to speech.
        model_path (str): Path to the pre-trained TTS model checkpoint (.pt file).
        data_cfg_path (str): Path to the data configuration file (data.yaml).

    Returns:
        tuple: A tuple containing the waveform tensor and the sample rate.
    """
    # Load the pre-trained TTS model and task
    models, cfg, task = load_model_ensemble_and_task(
        [model_path],
        arg_overrides={"data_config": data_cfg_path}
    )
    model = models[0]
    print("cfg:", cfg)
    inference = TTSHubInterface(cfg, task, model)
    # Build the generator for inference
    generator = task.build_generator([model], cfg)

    # Prepare the input text
    text_inputs = text.strip()
    inputs = inference.get_model_input(task, text_inputs)

    # Generate the speech waveform
    with torch.no_grad():
        waveform, sample_rate = inference.get_prediction(task, model, generator, inputs)

    return waveform, sample_rate

# Example usage:
# Replace 'path/to/tts_model.pt' and 'path/to/data.yaml' with your actual paths.
text_to_speak = "Hello, this is a test."
waveform, sr = generate_voice(text_to_speak, '/Users/rebelraider/.cache/huggingface/hub/models--Cyanbox--Prompt-Singer/snapshots/4a9ad215081865e168df26ea4a54cc78e87378a6/prompt-singer-flant5-large-finetuned/checkpoint_last.pt', 'huita.yaml')

# You can then save the waveform to an audio file using a library like librosa or soundfile.
# For example:
import soundfile as sf
sf.write('output.wav', waveform.numpy(), sr)

ModuleNotFoundError: No module named 'torch'

In [1]:
import logging
import sys

sys.path.insert(0, "Singer")
sys.path.insert(0, "research")
sys.path
import research
from fairseq import tasks
tasks.TASK_REGISTRY

/Users/rebelraider/Library/Caches/pypoetry/virtualenvs/xlabs-hack-2024-GK-D5AqO-py3.9/lib/python3.9/site-packages/fairscale/experimental/nn/offload.py:19: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  return torch.cuda.amp.custom_fwd(orig_func)  # type: ignore
/Users/rebelraider/Library/Caches/pypoetry/virtualenvs/xlabs-hack-2024-GK-D5AqO-py3.9/lib/python3.9/site-packages/fairscale/experimental/nn/offload.py:30: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  return torch.cuda.amp.custom_bwd(orig_func)  # type: ignore


{'sentence_prediction': fairseq.tasks.sentence_prediction.SentencePredictionTask,
 'sentence_prediction_adapters': fairseq.tasks.sentence_prediction_adapters.SentencePredictionAdapterTask,
 'speech_unit_modeling': fairseq.tasks.speech_ulm_task.SpeechUnitLanguageModelingTask,
 'hubert_pretraining': fairseq.tasks.hubert_pretraining.HubertPretrainingTask,
 'denoising': fairseq.tasks.denoising.DenoisingTask,
 'multilingual_denoising': fairseq.tasks.multilingual_denoising.MultilingualDenoisingTask,
 'translation': fairseq.tasks.translation.TranslationTask,
 'multilingual_translation': fairseq.tasks.multilingual_translation.MultilingualTranslationTask,
 'translation_from_pretrained_bart': fairseq.tasks.translation_from_pretrained_bart.TranslationFromPretrainedBARTTask,
 'translation_lev': fairseq.tasks.translation_lev.TranslationLevenshteinTask,
 'language_modeling': fairseq.tasks.language_modeling.LanguageModelingTask,
 'speech_to_text': fairseq.tasks.speech_to_text.SpeechToTextTask,
 'lega